# Exploratory Data Analysis

SQL-first validation of MetaPro RPKM sample data against taxonomy/pathway reference tables.

- [Design spec](../../../../docs/superpowers/specs/2026-06-11-exploratory-analysis-design.md)
- [Setup & workflow](../README.md)

In [1]:
from pathlib import Path

import duckdb

%load_ext sql

REPO_ROOT = Path("..").resolve().parents[1]  # notebooks → exploration → analytics → repo root
PARQUET_DIR = REPO_ROOT / "resources/db/parquet"
RPKM_FILES = [
    REPO_ROOT / "resources/example_data/test_rpkm_1.tsv",
    REPO_ROOT / "resources/example_data/test_rpkm_2.tsv",
]
TABLES = [
    "names", "nodes", "parents",
    "pathway_nodes", "pathway_edges",
    "pathway_superpathways", "superpathways",
]
KEY_COLS = ["GeneID", "Length", "Reads", "EC#", "RPKM", "Unclassified"]

conn = duckdb.connect()
%sql conn --alias duckdb

print(f"Repo root: {REPO_ROOT}")

The 'toml' package isn't installed. To load settings from pyproject.toml or ~/.jupysql/config, install with: pip install toml

Repo root: /Users/sibyl/study/metapro-data-vis/.worktrees/exploration-eda


In [2]:
import sys

missing = []
for table in TABLES:
    p = PARQUET_DIR / f"{table}.parquet"
    if not p.exists():
        missing.append(str(p))
for rpkm in RPKM_FILES:
    if not rpkm.exists():
        missing.append(str(rpkm))

if missing:
    print("MISSING INPUT FILES:")
    for m in missing:
        print(f"  - {m}")
    print("\nRun: uv run python exploration/scripts/export_parquet.py")
    sys.exit(1)

for table in TABLES:
    p = PARQUET_DIR / f"{table}.parquet"
    n = conn.execute(f"SELECT COUNT(*) FROM '{p}'").fetchone()[0]
    mb = p.stat().st_size / (1024 * 1024)
    print(f"{table}: {n:,} rows ({mb:.1f} MB)")

for rpkm in RPKM_FILES:
    mb = rpkm.stat().st_size / (1024 * 1024)
    print(f"{rpkm.name}: {mb:.1f} MB")

print("All inputs present.")

names: 2,840,139 rows (137.3 MB)
nodes: 2,840,139 rows (10.8 MB)
parents: 2,840,134 rows (129.7 MB)
pathway_nodes: 23,864 rows (1.0 MB)
pathway_edges: 46,726 rows (2.5 MB)
pathway_superpathways: 193 rows (0.0 MB)
superpathways: 13 rows (0.0 MB)
test_rpkm_1.tsv: 51.0 MB
test_rpkm_2.tsv: 71.1 MB
All inputs present.


## 3. Data Dictionary

Register the exported reference Parquet files as DuckDB views, then inspect schemas and row counts for all tables in scope.

In [3]:
for table in TABLES:
    conn.execute(f"""
        CREATE OR REPLACE VIEW {table} AS
        SELECT * FROM '{PARQUET_DIR / f"{table}.parquet"}'
    """)

print("Views created for all reference tables.")

Views created for all reference tables.


In [4]:
%%sql
SELECT 'names' AS tbl, column_name, column_type FROM (DESCRIBE names)
UNION ALL SELECT 'nodes', column_name, column_type FROM (DESCRIBE nodes)
UNION ALL SELECT 'parents', column_name, column_type FROM (DESCRIBE parents)
UNION ALL SELECT 'pathway_nodes', column_name, column_type FROM (DESCRIBE pathway_nodes)
UNION ALL SELECT 'pathway_edges', column_name, column_type FROM (DESCRIBE pathway_edges)
UNION ALL SELECT 'pathway_superpathways', column_name, column_type FROM (DESCRIBE pathway_superpathways)
UNION ALL SELECT 'superpathways', column_name, column_type FROM (DESCRIBE superpathways)
ORDER BY tbl, column_name;

Running query in 'duckdb'

tbl,column_name,column_type
names,id,VARCHAR
names,name,VARCHAR
names,tax_id,BIGINT
nodes,id,BIGINT
parents,id,VARCHAR
parents,t_class,BIGINT
parents,t_family,BIGINT
parents,t_genus,BIGINT
parents,t_kingdom,BIGINT
parents,t_order,BIGINT


In [5]:
%%sql
SELECT 'names' AS tbl, COUNT(*) AS n FROM names
UNION ALL SELECT 'nodes', COUNT(*) FROM nodes
UNION ALL SELECT 'parents', COUNT(*) FROM parents
UNION ALL SELECT 'pathway_nodes', COUNT(*) FROM pathway_nodes
UNION ALL SELECT 'pathway_edges', COUNT(*) FROM pathway_edges
UNION ALL SELECT 'pathway_superpathways', COUNT(*) FROM pathway_superpathways
UNION ALL SELECT 'superpathways', COUNT(*) FROM superpathways
ORDER BY tbl;

Running query in 'duckdb'

tbl,n
names,2840139
nodes,2840139
parents,2840134
pathway_edges,46726
pathway_nodes,23864
pathway_superpathways,193
superpathways,13


## 4. Cardinality

Measure basic table cardinalities and graph degree distributions that downstream joins rely on.

In [6]:
%%sql
SELECT
    MIN(name_count) AS min_names,
    MAX(name_count) AS max_names,
    MEDIAN(name_count) AS median_names,
    AVG(name_count) AS avg_names
FROM (
    SELECT tax_id, COUNT(*) AS name_count
    FROM names
    GROUP BY tax_id
);

Running query in 'duckdb'

min_names,max_names,median_names,avg_names
1,1,1.0,1.0


In [7]:
%%sql
WITH out_degree AS (
    SELECT source AS node_id, COUNT(*) AS out_deg
    FROM pathway_edges
    GROUP BY source
),
in_degree AS (
    SELECT target AS node_id, COUNT(*) AS in_deg
    FROM pathway_edges
    GROUP BY target
),
all_nodes AS (
    SELECT id AS node_id FROM pathway_nodes
)
SELECT
    'out_degree' AS metric,
    MIN(COALESCE(o.out_deg, 0)) AS min,
    MAX(COALESCE(o.out_deg, 0)) AS max,
    MEDIAN(COALESCE(o.out_deg, 0)) AS median
FROM all_nodes n LEFT JOIN out_degree o ON n.node_id = o.node_id
UNION ALL
SELECT
    'in_degree',
    MIN(COALESCE(i.in_deg, 0)),
    MAX(COALESCE(i.in_deg, 0)),
    MEDIAN(COALESCE(i.in_deg, 0))
FROM all_nodes n LEFT JOIN in_degree i ON n.node_id = i.node_id;

Running query in 'duckdb'

metric,min,max,median
out_degree,0,945,0.0
in_degree,0,945,0.0


In [8]:
%%sql
SELECT
    MIN(edge_count) AS min_edges,
    MAX(edge_count) AS max_edges,
    MEDIAN(edge_count) AS median_edges
FROM (
    SELECT pathway, COUNT(*) AS edge_count
    FROM pathway_edges
    GROUP BY pathway
);

Running query in 'duckdb'

min_edges,max_edges,median_edges
2,2410,132.0


## 5. RPKM Profiling

Load both sample RPKM TSV files as DuckDB views, normalize EC values, and profile tax_id column coverage and sparsity.

In [9]:
for i, rpkm_path in enumerate(RPKM_FILES, start=1):
    conn.execute(f"""
        CREATE OR REPLACE VIEW rpkm_{i} AS
        SELECT * FROM read_csv('{rpkm_path}', delim='\t', header=true, auto_detect=true)
    """)
    n = conn.execute(f"SELECT COUNT(*) FROM rpkm_{i}").fetchone()[0]
    print(f"rpkm_{i} ({rpkm_path.name}): {n:,} rows")

rpkm_1 (test_rpkm_1.tsv): 425,828 rows
rpkm_2 (test_rpkm_2.tsv): 461,112 rows


In [10]:
%%sql
SELECT
    "EC#" AS ec_raw,
    CASE
        WHEN "EC#" IS NULL OR TRIM(CAST("EC#" AS VARCHAR)) IN ('', 'None', 'none', 'NA', 'null') THEN '0.0.0.0'
        WHEN STARTS_WITH(CAST("EC#" AS VARCHAR), 'EC:') THEN SUBSTR(CAST("EC#" AS VARCHAR), 4)
        ELSE CAST("EC#" AS VARCHAR)
    END AS ec_normalized,
    COUNT(*) AS row_count
FROM rpkm_1
GROUP BY 1, 2
ORDER BY row_count DESC
LIMIT 20;

Running query in 'duckdb'

ec_raw,ec_normalized,row_count
None,0.0.0.0,235991
EC:2.7.13.3,2.7.13.3,8513
EC:3.6.4.12,3.6.4.12,4279
EC:2.7.7.6,2.7.7.6,3466
EC:2.3.2.27,2.3.2.27,2987
EC:2.7.11.1,2.7.11.1,2672
EC:2.7.7.7,2.7.7.7,2658
EC:3.6.3.14,3.6.3.14,2645
EC:2.7.1.69,2.7.1.69,2412
EC:5.2.1.8,5.2.1.8,2128


In [11]:
cols = conn.execute("SELECT * FROM rpkm_1 LIMIT 0").description
all_cols = [c[0] for c in cols]
tax_cols = [c for c in all_cols if c not in KEY_COLS]
print(f"Tax_id columns: {len(tax_cols)}")

# DuckDB UNPIVOT needs the dynamic tax_id column list generated in Python.
tax_cols_sql = ", ".join(f'"{c}"' for c in tax_cols)
conn.execute(f"""
    CREATE OR REPLACE VIEW rpkm_1_long AS
    UNPIVOT rpkm_1
    ON {tax_cols_sql}
    INTO NAME tax_id VALUE rpkm_value
""")
print("rpkm_1 sparsity:", conn.execute("""
    SELECT
        COUNT(*) AS total_cells,
        COUNT(CASE WHEN rpkm_value > 0 THEN 1 END) AS nonzero_cells,
        ROUND(100.0 * COUNT(CASE WHEN rpkm_value > 0 THEN 1 END) / COUNT(*), 2) AS pct_nonzero
    FROM rpkm_1_long
""").fetchone())

Tax_id columns: 8


rpkm_1 sparsity: (3406624, 59439, 1.74)


In [12]:
tax_set_1 = set(tax_cols)
cols2 = [c[0] for c in conn.execute("SELECT * FROM rpkm_2 LIMIT 0").description]
tax_set_2 = set(c for c in cols2 if c not in KEY_COLS)
print(f"rpkm_1 tax columns: {len(tax_set_1)}")
print(f"rpkm_2 tax columns: {len(tax_set_2)}")
print(f"intersection: {len(tax_set_1 & tax_set_2)}")
print(f"only in rpkm_1: {len(tax_set_1 - tax_set_2)}")
print(f"only in rpkm_2: {len(tax_set_2 - tax_set_1)}")

rpkm_1 tax columns: 8
rpkm_2 tax columns: 12
intersection: 6
only in rpkm_1: 2
only in rpkm_2: 6


## 6. Reference Integrity

Check internal reference-table relationships, duplicate pathway EC names, rank completeness, and pathway nodes without graph edges.

In [13]:
%%sql
SELECT 'names.tax_id -> nodes' AS fk,
       COUNT(*) AS orphan_count
FROM names n
LEFT JOIN nodes nd ON n.tax_id = nd.id
WHERE nd.id IS NULL
UNION ALL
SELECT 'parents.t_kingdom -> nodes', COUNT(*)
FROM parents p LEFT JOIN nodes nd ON p.t_kingdom = nd.id
WHERE p.t_kingdom IS NOT NULL AND nd.id IS NULL
UNION ALL
SELECT 'pathway_edges.source -> pathway_nodes', COUNT(*)
FROM pathway_edges e LEFT JOIN pathway_nodes pn ON e.source = pn.id
WHERE pn.id IS NULL
UNION ALL
SELECT 'pathway_superpathways.superpathway -> superpathways', COUNT(*)
FROM pathway_superpathways ps LEFT JOIN superpathways sp ON ps.superpathway = sp.id
WHERE sp.id IS NULL;

Running query in 'duckdb'

fk,orphan_count
names.tax_id -> nodes,0
parents.t_kingdom -> nodes,0
pathway_edges.source -> pathway_nodes,0
pathway_superpathways.superpathway -> superpathways,0


In [14]:
%%sql
SELECT name, COUNT(*) AS n
FROM pathway_nodes
GROUP BY name
HAVING COUNT(*) > 1
ORDER BY n DESC
LIMIT 20;

Running query in 'duckdb'

name,n
1.14.14.1,77
Glycolysis / Gluconeogenesis,49
C00022,48
C00024,48
Citrate cycle (TCA cycle),46
Pyruvate metabolism,35
C00083,35
4.2.1.17,33
2.3.1.85,33
"Alanine, aspartate and glutamate metabolism",32


In [15]:
%%sql
SELECT
    COUNT(*) AS total,
    COUNT(CASE WHEN t_genus IS NOT NULL AND t_family IS NULL THEN 1 END) AS genus_without_family,
    COUNT(CASE WHEN t_genus IS NOT NULL AND t_order IS NULL THEN 1 END) AS genus_without_order,
    COUNT(CASE WHEN t_phylum IS NOT NULL AND t_kingdom IS NULL THEN 1 END) AS phylum_without_kingdom
FROM parents;

Running query in 'duckdb'

total,genus_without_family,genus_without_order,phylum_without_kingdom
2840134,0,0,0


In [16]:
%%sql
SELECT pn.pathway, COUNT(*) AS dangling_nodes
FROM pathway_nodes pn
LEFT JOIN pathway_edges e ON pn.id = e.source OR pn.id = e.target
WHERE e.id IS NULL
GROUP BY pn.pathway
ORDER BY dangling_nodes DESC
LIMIT 10;

Running query in 'duckdb'

pathway,dangling_nodes
1100,3716
1110,2480
1120,1208
1040,135
999,128
1057,127
980,106
998,100
904,99
624,97


## 7. Cross-Domain Joins

Validate joins from RPKM tax_id headers and EC values into the taxonomy and pathway reference tables.

In [17]:
def materialize_rpk_tax_ids(tax_ids: set[str]) -> None:
    ids_int = [int(t) for t in tax_ids]
    # Use a regular session table so the following %%sql cell can see it.
    conn.execute("CREATE OR REPLACE TABLE rpk_tax_ids (tax_id INT)")
    conn.executemany("INSERT INTO rpk_tax_ids VALUES (?)", [(i,) for i in ids_int])


def tax_id_match_rates(tax_ids: set[str], label: str) -> None:
    materialize_rpk_tax_ids(tax_ids)
    r = conn.execute("""
        SELECT
            (SELECT COUNT(*) FROM rpk_tax_ids) AS total_headers,
            (SELECT COUNT(*) FROM rpk_tax_ids r JOIN names n ON r.tax_id = n.tax_id) AS matched_names,
            (SELECT COUNT(DISTINCT r.tax_id) FROM rpk_tax_ids r JOIN names n ON r.tax_id = n.tax_id) AS distinct_tax_ids_with_names,
            (SELECT COUNT(*) FROM rpk_tax_ids r JOIN nodes nd ON r.tax_id = nd.id) AS matched_nodes,
            (SELECT COUNT(*) FROM rpk_tax_ids r JOIN parents p ON r.tax_id = p.tax_id) AS matched_parents
    """).fetchone()
    print(f"--- {label} ---")
    print(f"  tax_id column headers: {r[0]}")
    print(f"  headers with >=1 names row: {r[1]} (distinct tax_ids: {r[2]})")
    print(f"  headers with nodes row: {r[3]}")
    print(f"  headers with parents row: {r[4]}")


tax_id_match_rates(tax_set_1, "test_rpkm_1")
tax_id_match_rates(tax_set_2, "test_rpkm_2")
materialize_rpk_tax_ids(tax_set_1)
print("rpk_tax_ids reset to test_rpkm_1 for the unmapped tax_id sample below.")

--- test_rpkm_1 ---
  tax_id column headers: 8
  headers with >=1 names row: 8 (distinct tax_ids: 8)
  headers with nodes row: 8
  headers with parents row: 8
--- test_rpkm_2 ---
  tax_id column headers: 12
  headers with >=1 names row: 12 (distinct tax_ids: 12)
  headers with nodes row: 12
  headers with parents row: 12
rpk_tax_ids reset to test_rpkm_1 for the unmapped tax_id sample below.


In [18]:
%%sql
SELECT r.tax_id
FROM rpk_tax_ids r
LEFT JOIN names n ON r.tax_id = n.tax_id
WHERE n.tax_id IS NULL
ORDER BY r.tax_id
LIMIT 20;

Running query in 'duckdb'

tax_id


In [19]:
%%sql
WITH normalized AS (
    SELECT DISTINCT
        CASE
            WHEN "EC#" IS NULL OR TRIM(CAST("EC#" AS VARCHAR)) IN ('', 'None', 'none', 'NA', 'null') THEN '0.0.0.0'
            WHEN STARTS_WITH(CAST("EC#" AS VARCHAR), 'EC:') THEN SUBSTR(CAST("EC#" AS VARCHAR), 4)
            ELSE CAST("EC#" AS VARCHAR)
        END AS ec
    FROM rpkm_1
)
SELECT
    (SELECT COUNT(*) FROM normalized) AS distinct_ecs,
    (SELECT COUNT(*) FROM normalized n JOIN pathway_nodes pn ON n.ec = pn.name) AS matched_pathway_nodes;

Running query in 'duckdb'

distinct_ecs,matched_pathway_nodes
9034,3258


In [20]:
%%sql
SELECT pn.name AS ec, pn.pathway, psp.id AS psp_id, sp.id AS sp_id
FROM pathway_nodes pn
LEFT JOIN pathway_superpathways psp ON pn.pathway = psp.id
LEFT JOIN superpathways sp ON psp.superpathway = sp.id
WHERE psp.id IS NULL OR sp.id IS NULL
LIMIT 20;

Running query in 'duckdb'

ec,pathway,psp_id,sp_id


## 8. Findings Summary

### Validated assumptions

- Parquet reference inputs are present and readable: `names` has 2,840,139 rows, `nodes` has 2,840,139 rows, `parents` has 2,840,134 rows, `pathway_nodes` has 23,864 rows, `pathway_edges` has 46,726 rows, `pathway_superpathways` has 193 rows, and `superpathways` has 13 rows.
- RPKM files load as wide tables with fixed columns `GeneID`, `Length`, `Reads`, `EC#`, `RPKM`, and `Unclassified`, followed by tax_id columns. `test_rpkm_1.tsv` has 425,828 rows and 8 tax_id columns; `test_rpkm_2.tsv` has 461,112 rows and 12 tax_id columns.
- The two sample RPKM files share 6 tax_id columns; 2 appear only in `test_rpkm_1.tsv` and 6 appear only in `test_rpkm_2.tsv`.
- Tax_id column headers resolve completely in the reference taxonomy for both sample files: `test_rpkm_1.tsv` has 8/8 headers with `names`, `nodes`, and `parents` matches; `test_rpkm_2.tsv` has 12/12 headers with `names`, `nodes`, and `parents` matches.
- The sampled internal reference joins checked here have 0 observed orphans for `names.tax_id -> nodes`, `parents.t_kingdom -> nodes`, `pathway_edges.source -> pathway_nodes`, and `pathway_superpathways.superpathway -> superpathways`.
- The EC-to-superpathway chain checked here has no observed missing `pathway_superpathways` or `superpathways` links in the first 20 rows returned by the missing-link query.

### Discrepancies vs logical model (see spec Section 14)

| Edge | Expected | Observed | Problem? |
|---|---|---|---|
| RPKM -> `names` | 0..many `names` rows per tax_id header; multiple names per tax_id expected | Header coverage is 8/8 for `test_rpkm_1.tsv` and 12/12 for `test_rpkm_2.tsv`; reference `names` cardinality by `tax_id` is min 1, max 1, median 1.0, avg 1.0 | Review: observed sample has one `names` row per tax_id, not multiple synonyms |
| RPKM -> `nodes` | 0..1 `nodes` row per tax_id header | Header coverage is 8/8 for `test_rpkm_1.tsv` and 12/12 for `test_rpkm_2.tsv` | No discrepancy observed in sample |
| RPKM -> `parents` | 0..1 optional `parents` row per tax_id header | Header coverage is 8/8 for `test_rpkm_1.tsv` and 12/12 for `test_rpkm_2.tsv` | No discrepancy observed in sample; optionality remains possible outside these samples |
| RPKM `EC#` -> `pathway_nodes.name` | 0..many `pathway_nodes` rows per normalized EC value | In `test_rpkm_1.tsv`, 9,034 distinct normalized EC values produce 3,258 matched `pathway_nodes` rows | Review unmatched normalized ECs and expected coverage |
| `pathway_nodes.pathway` -> `pathway_superpathways.id` -> `superpathways.id` | 0..1 chain per matched pathway node | Missing-link query returned 0 rows in the displayed sample | No discrepancy observed in displayed output |
| `nodes` -> `names` | One node can have many names | `names` cardinality by `tax_id` is exactly 1 in this dump | Review: logical many side is not exercised by current reference data |

### Gotchas for future analytics

- Treat RPKM tax_id columns as wide-measure headers, not row fields; joins to taxonomy are per header.
- Normalize `EC#` before joining to pathway nodes: `EC:x.y.z` becomes `x.y.z`, while null/`None`-like values become `0.0.0.0`. In `test_rpkm_1.tsv`, `None` normalizes to `0.0.0.0` for 235,991 rows.
- RPKM abundance is sparse in wide form: `test_rpkm_1.tsv` has 3,406,624 tax_id cells, 59,439 nonzero cells, and 1.74% nonzero coverage.
- `pathway_nodes.name` is not unique. The top duplicate shown is `1.14.14.1` with 77 rows, so EC joins can fan out.
- Pathway graph degree distributions are highly skewed: both in-degree and out-degree have min 0, max 945, median 0.0; dangling pathway nodes are present in the displayed output, led by pathway 1100 with 3,716 dangling nodes.
- `pathway_nodes.pathway -> pathway_superpathways.id` is a logical relationship used by the app, not a declared SQLite foreign key.
- The notebook outputs are display-limited in some sections; use the underlying SQL cells for deeper audits rather than assuming visible top-N tables are exhaustive.